In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import mlflow
import re
def nettoyer_texte(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r'<.*?>', '', text)   # HTML
    text = re.sub(r'\s+', ' ', text)    # Espaces multiples
    return text

In [2]:
df = pd.read_csv("../data/IMDB Dataset.csv", engine="python", on_bad_lines="skip")
df['review'] = df['review'].apply(nettoyer_texte)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:

analyzer = SentimentIntensityAnalyzer()

df["compound"] = df["review"].apply(lambda x: analyzer.polarity_scores(x)["compound"])
df["vader_sentiment"] = df["compound"].apply(
    lambda x: "positive" if x > 0.1 else ("negative" if x < -0.1 else "neutral")
)

df_eval = df[df["vader_sentiment"] != "neutral"]
accuracy = (df_eval["sentiment"] == df_eval["vader_sentiment"]).mean()

# MLflow
mlflow.set_experiment("VADER_IMDB") 

with mlflow.start_run():
    mlflow.log_param("threshold_pos", 0.1)
    mlflow.log_param("threshold_neg", -0.1)
    mlflow.log_metric("accuracy", accuracy)

    df_eval.head(50).to_csv("vader_results_sample.csv", index=False)
    mlflow.log_artifact("vader_results_sample.csv")

print("Accuracy VADER :", accuracy)
print(df_eval[["sentiment", "vader_sentiment", "compound"]].head())


KeyboardInterrupt: 